# quaero_mlp

Multi-head Gumbel subset-sampling model — **MLP backbone, fully graph-free**.

One head per target bin.  A shared per-node **MLP consumes the descriptor
coordinates directly** — there is no graph anywhere.  Head $k$ maps the
embedding of bin-$k$ nodes to logits and selects **exactly** $b_k$ of them
via relaxed Gumbel top-$k$ subset sampling (Xie & Ermon, 2019 — successive
softmax).  The per-bin quotas $b_k$ come from `find_sampling_plan`, so the
quota constraint is satisfied *by construction* — no penalty terms.

**Sequential selection with a running Gaussian.**  Bins are processed one at
a time, ordered by **descending requirement $b_k$**.  After each bin, the
running **mean and covariance** of everything selected so far define a
Gaussian; the next bin's head is conditioned on each candidate's squared
**Mahalanobis distance** to that Gaussian (a scalar carrying both mean and
covariance), plus a has-context flag.  The largest-requirement bin runs first
with no Gaussian and anchors it; every subsequent bin favours nodes that fit
the running Gaussian.  Fully differentiable — the running moments come from
earlier bins' soft selections (first/second-moment accumulators), so
gradients couple the heads.

**Node features are descriptors only.**

The demo dataset plants a ground-truth subgroup: descriptors are drawn from
`n_blobs` random Gaussian clusters, and one whole blob has its target
redrawn from $\mathcal{N}(0.5, 0.15)$ — the specified distribution — giving
a genuine dense cluster the model should recover.

## Loss functions (three, selectable in the train loop)

**1. Soft within-cluster scatter (active).**  Membership-weighted mean
squared distance of the selection to its centroid:

$$\mathcal{L} = \frac{\sum_i x_i \, \lVert p_i - \mu \rVert^2}{\sum_i x_i},
\qquad \mu = \frac{\sum_i x_i p_i}{\sum_i x_i}$$

**Dense gradient** — every selected node is pulled toward the centroid in
proportion to its squared distance, so the model gets a strong, non-sparse
signal to collapse the selection into a tight cluster.  (The box-ratio loss
below gives only boundary gradients and trains poorly on its own.)

**2. Soft hyperrectangle ratio.**  Maximise selected / nodes-inside-the-box:

$$\mathcal{L} = -\frac{\sum_i x_i}{\sum_i s_i}, \qquad
s_i = \prod_d \sigma\!\left(\tfrac{p_{i,d} - \mathrm{lo}_d}{T}\right)
\sigma\!\left(\tfrac{\mathrm{hi}_d - p_{i,d}}{T}\right)$$

Bounds via membership-weighted soft-min/max; inside test via per-dimension
sigmoids.  Gradient reaches only the extreme boundary nodes.

**3. Soft hyperellipsoid member-in maximisation.**  Fit a Gaussian to the
selection and maximise the (soft) number of members inside its $q$-level
ellipsoid; affine-invariant, so it rewards shape but not absolute scale.

Swap the active loss via the commented lines in the train loop.

In [ ]:
# !pip install plotly
# (MLP model is graph-free -- no torch_geometric / pykeops needed)

In [ ]:
import pandas as pd, numpy as np, math
from scipy import stats as scipy_stats
from typing import Union, List, Tuple, Callable
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

In [ ]:
def bin_target(df: pd.DataFrame, target_col: str) -> dict:
    """
    Discretise a target column using Scott's rule and return a mapping
    of bin_id -> list of row indices.

    Scott's rule:
        bin_width  h = 3.5 * std(x) * n^(-1/3)
        n_bins       = ceil((max(x) - min(x)) / h)
    """
    x = df[target_col].dropna().values
    n = len(x)

    if n < 2:
        raise ValueError("Need at least 2 non-null values to compute bins.")

    std = np.std(x, ddof=1)          # sample standard deviation
    if std == 0:
        raise ValueError("Target column has zero variance; cannot bin.")

    h       = 3.5 * std * (n ** (-1 / 3))          # Scott's bin width
    n_bins  = max(1, math.ceil((x.max() - x.min()) / h))

    bin_series = pd.cut(
        df[target_col],
        bins=n_bins,
        labels=range(n_bins),
        include_lowest=True
    )

    bin_dict = {}
    for bin_id, group in df.groupby(bin_series, observed=True):
        bin_dict[int(bin_id)] = list(group.index)

    return bin_dict, n_bins

In [ ]:
DIST_MAP = {
    "normal":      scipy_stats.norm,
    "gaussian":    scipy_stats.norm,
    "exponential": scipy_stats.expon,
    "beta":        scipy_stats.beta,
    "gamma":       scipy_stats.gamma,
    "uniform":     scipy_stats.uniform,
    "lognormal":   scipy_stats.lognorm,
}

def make_pdf(
    spec: Union[
        Tuple[str, dict],
        List[Tuple[str, dict, float]]
    ]
) -> Callable:
    """
    Single distribution:
        spec = ("normal", {"loc": 0, "scale": 1})

    Mixture (weights are auto-normalised):
        spec = [
            ("normal",      {"loc": 0, "scale": 1},  0.6),
            ("exponential", {"scale": 2},             0.4),
        ]
    """
    if isinstance(spec, tuple):
        name, params = spec
        dist = DIST_MAP[name.lower()]
        return lambda x: dist.pdf(x, **params)

    total_w    = sum(w for _, _, w in spec)
    components = [(DIST_MAP[n.lower()], p, w / total_w) for n, p, w in spec]

    def mixture_pdf(x):
        return sum(w * d.pdf(x, **p) for d, p, w in components)

    return mixture_pdf


def _bin_centers(df: pd.DataFrame, target_col: str, n_bins: int) -> np.ndarray:
    """Midpoint of each equal-width bin over the target column range."""
    lo, hi = df[target_col].min(), df[target_col].max()
    edges  = np.linspace(lo, hi, n_bins + 1)
    return (edges[:-1] + edges[1:]) / 2

In [ ]:
EPS = 1e-9   # prevents bins from collapsing to 0 when c = 0

def find_sampling_plan(
    bin_map:       dict,
    df:            pd.DataFrame,
    target_col:    str,
    n_bins:        int,
    pdf:           Callable,
    min_total_pct: float = 0.0,
) -> Tuple[dict, List[int], bool]:
    """
    Trough-anchored iteration: for c = 0, 1, ..., capacity(anchor bin),
    derive required counts for every bin as round((c + eps) * pdf_i /
    pdf_anchor).  Bins that were anchors in earlier (failed) rounds are
    clamped to their capacity and never block feasibility.  Picks the level
    whose total is the smallest value >= min_total_pct * len(df).

    Returns (required {bin_id: b_k}, peak_bins, feasible).
    """
    centers  = _bin_centers(df, target_col, n_bins)
    bin_ids  = sorted(bin_map.keys())
    pdf_vals = {bid: pdf(centers[bid]) for bid in bin_ids}

    peak_pdf  = max(pdf_vals.values())
    peak_bins = [bid for bid in bin_ids if np.isclose(pdf_vals[bid], peak_pdf)]

    min_total = round(min_total_pct * len(df))

    seen    = set()  # bins that were anchors in earlier, infeasible rounds
    anchors = sorted(bin_ids, key=lambda b: pdf_vals[b])

    for anchor in anchors:
        anchor_pdf = pdf_vals[anchor]
        anchor_cap = len(bin_map[anchor])

        feasible_levels = []
        for c in range(anchor_cap + 1):
            required = {}
            for bid in bin_ids:
                val = round((c + EPS) * pdf_vals[bid] / anchor_pdf)
                required[bid] = min(val, len(bin_map[bid])) if bid in seen else val
            if all(required[bid] <= len(bin_map[bid]) for bid in bin_ids):
                if sum(required.values()) >= min_total:
                    feasible_levels.append((sum(required.values()), required))

        if feasible_levels:
            _, best_required = min(feasible_levels, key=lambda x: x[0])
            return best_required, peak_bins, True

        seen.add(anchor)

    # every bin used as an anchor -- the only possible level is "take everything"
    required = {bid: len(bin_map[bid]) for bid in bin_ids}
    if sum(required.values()) >= min_total:
        return required, peak_bins, True
    return {}, peak_bins, False

In [ ]:
np.random.seed(42)
rng = np.random.default_rng(42)

N        = 1500
n_blobs  = 4
blob_std = 0.08

# random Gaussian blob descriptors (clustered, non-uniform)
centers = rng.uniform(0.0, 1.0, size=(n_blobs, 3))
assign  = rng.integers(0, n_blobs, size=N)                 # blob membership per point
coords  = centers[assign] + rng.normal(0.0, blob_std, size=(N, 3))

df = pd.DataFrame(coords, columns=["feature_a", "feature_b", "feature_c"])
df["target"] = rng.random(N)                               # background target ~ Uniform[0, 1)

# plant a subgroup: one whole blob gets target ~ N(0.5, 0.15) (the specified distribution)
planted_blob = 0
condition = pd.Series(assign == planted_blob)
print(condition.sum())
df.loc[condition, "target"] = rng.normal(loc=0.5, scale=0.15, size=int(condition.sum()))

target_col = "target"
desc_cols  = [c for c in df.columns if c != target_col]
N          = len(df)

bin_map, n_bins = bin_target(df, target_col)      # Scott's rule

# THE GIVEN DISTRIBUTION and the per-bin quotas b_k (model inputs)
pdf = make_pdf(("gaussian", {"loc": 0.5, "scale": 0.15}))
required, peak_bins, feasible = find_sampling_plan(
    bin_map, df, target_col, n_bins, pdf, min_total_pct=0.10,
)
print(f"feasible={feasible}  peak_bins={peak_bins}")
print(f"required: {required}")
print(f"total required: {sum(required.values())} / {N}")

# descriptor coordinates = node features (no graph, no target histogram)
pos   = torch.tensor(df[desc_cols].to_numpy(), dtype=torch.float32, device=device)
feats = pos

# per-bin node index tensors
bin_node_idx = {bid: torch.tensor(idxs, device=device, dtype=torch.long)
                for bid, idxs in bin_map.items()}

print(f"n_bins={n_bins}  feats={tuple(feats.shape)}")

In [ ]:
def make_mlp(*dims):
    """Linear -> ReLU -> ... -> Linear (no activation on last layer)."""
    layers = []
    for i in range(len(dims) - 1):
        layers.append(nn.Linear(dims[i], dims[i + 1]))
        if i < len(dims) - 2:
            layers.append(nn.ReLU())
    return nn.Sequential(*layers)


def gumbel_subset_sample(logits, k, tau=1.0, hard=False, eps=1e-20):
    """
    Relaxed top-k subset sampling (Xie & Ermon, 2019 -- successive softmax).

    logits : (n,) unnormalised scores for the n candidate nodes
    k      : number of items to select
    tau    : temperature (annealed during training)
    hard   : straight-through -- exact k-hot forward, soft gradients backward

    Returns x : (n,) soft k-hot vector with sum(x) = k, differentiable
    w.r.t. logits.  Gumbel noise is added once; each of the k rounds
    suppresses the mass already selected and takes a softmax, so gradients
    flow through every round.
    """
    g = -torch.log(-torch.log(torch.rand_like(logits) + eps) + eps)
    scores = (logits + g) / tau

    x = torch.zeros_like(logits)
    onehot_approx = torch.zeros_like(logits)
    for _ in range(k):
        mask   = torch.clamp(1.0 - onehot_approx, min=eps)   # suppress selected mass
        scores = scores + torch.log(mask)
        onehot_approx = F.softmax(scores, dim=0)
        x = x + onehot_approx

    if hard:
        idx    = torch.topk(x, k).indices
        x_hard = torch.zeros_like(x).scatter_(0, idx, 1.0)
        x = (x_hard - x).detach() + x
    return x

In [ ]:
class MultiHeadSubsetMLP(nn.Module):
    """
    One head per target bin.  A shared per-node MLP consumes the descriptor
    coordinates directly (no graph, no message passing) to embed each node.

    Bins are processed SEQUENTIALLY, ordered by descending requirement b_k.
    After each bin, the running mean and covariance of everything selected so
    far define a Gaussian; the next bin's head is conditioned on each
    candidate's squared Mahalanobis distance to that Gaussian (a scalar that
    carries both the mean and the covariance).  The largest-requirement bin
    runs first with no Gaussian and anchors it; subsequent bins favour nodes
    that fit the running Gaussian (low Mahalanobis).

    Head k maps [embedding | maha^2-to-running-Gaussian | has-context flag]
    to logits and Gumbel-samples exactly b_k nodes; the per-bin soft
    selections are scattered into a global (N,) vector, so sum(selection) =
    sum_k b_k by construction.

    Fully differentiable: the running mean/covariance come from earlier bins'
    soft selections (via first/second-moment accumulators), so gradients
    couple the heads.
    """
    def __init__(self, in_channels, hidden_channels, num_mlp_layers, n_bins, ell_reg=1e-4):
        super().__init__()
        self.ell_reg = ell_reg
        # shared per-node encoder MLP over the descriptors
        enc_dims = [in_channels] + [hidden_channels] * num_mlp_layers
        self.encoder = make_mlp(*enc_dims)
        # head input = node embedding | maha^2 (1) | has-context flag (1)
        self.heads = nn.ModuleList([
            make_mlp(hidden_channels + 2, hidden_channels, 1) for _ in range(n_bins)
        ])

    def forward(self, feats, pos, bin_node_idx, required, tau=1.0, hard=False, eps=1e-9):
        """
        feats        : (N, in_channels)  full node feature tensor (descriptors)
        pos          : (N, Dc)           descriptor coordinates (for the Gaussian)
        bin_node_idx : {bin_id: LongTensor of node indices in that bin}
        required     : {bin_id: b_k}     per-bin quotas (model inputs)

        Returns selection (N,), logits_all (N,)
        """
        N, Dc = feats.size(0), pos.size(1)
        selection  = torch.zeros(N, device=feats.device)
        logits_all = torch.zeros(N, device=feats.device)

        # largest-requirement bin first, down to the smallest
        order = sorted(bin_node_idx.keys(), key=lambda b: required.get(b, 0), reverse=True)

        # running (soft) moments of everything selected so far
        w_sum = torch.zeros((), device=feats.device)      # sum x
        s1    = torch.zeros(Dc, device=feats.device)      # sum x p
        s2    = torch.zeros(Dc, Dc, device=feats.device)  # sum x p p^T
        eye   = torch.eye(Dc, device=feats.device)

        for bid in order:
            idx = bin_node_idx[bid]
            b_k = int(required.get(bid, 0))
            if b_k == 0 or idx.numel() == 0:
                continue

            h = F.relu(self.encoder(feats[idx]))          # (n_k, hidden)
            P = pos[idx]                                  # (n_k, Dc)

            if float(w_sum) > 0:                          # running Gaussian exists
                mu    = s1 / (w_sum + eps)
                Sigma = s2 / (w_sum + eps) - torch.outer(mu, mu) + self.ell_reg * eye
                L     = torch.linalg.cholesky(Sigma)
                z     = torch.linalg.solve_triangular(L, (P - mu).T, upper=False)  # (Dc, n_k)
                maha2 = (z ** 2).sum(0)                   # (n_k,)
                has   = torch.ones(idx.numel(), 1, device=feats.device)
            else:
                maha2 = torch.zeros(idx.numel(), device=feats.device)
                has   = torch.zeros(idx.numel(), 1, device=feats.device)

            ctx    = torch.cat([maha2.unsqueeze(1), has], dim=1)     # (n_k, 2)
            logits = self.heads[bid](torch.cat([h, ctx], dim=1)).squeeze(-1)
            logits_all[idx] = logits

            x_k = gumbel_subset_sample(logits, min(b_k, idx.numel()), tau=tau, hard=hard)
            selection = selection.index_add(0, idx, x_k)

            # update running moments with this bin's soft selection
            xp    = x_k.unsqueeze(1) * P                  # (n_k, Dc)
            w_sum = w_sum + x_k.sum()
            s1    = s1 + xp.sum(0)
            s2    = s2 + xp.T @ P                         # sum x p p^T

        return selection, logits_all

In [ ]:
def soft_scatter_loss(selection, pos, eps=1e-9):
    """
    Membership-weighted within-cluster scatter: mean squared distance of the
    (softly) selected nodes to their weighted centroid.

        loss = sum_i x_i * ||p_i - mu||^2 / sum_i x_i,   mu = weighted mean

    Dense gradient -- every selected node is pulled toward the centroid in
    proportion to its squared distance, so the model gets a strong signal to
    collapse the selection into a tight cluster (unlike the box ratio, whose
    gradient reaches only the few extreme boundary nodes).  Not
    affine-invariant, so it does pressure absolute scale.

    selection : (N,) soft selection values in [0, 1]
    pos       : (N, D) descriptor coordinates
    """
    w  = selection / (selection.sum() + eps)          # (N,)
    mu = (w.unsqueeze(1) * pos).sum(0)                 # (D,)
    d2 = ((pos - mu) ** 2).sum(1)                      # (N,)
    return (w * d2).sum()


def soft_hyperrectangle_ratio_loss(selection, pos, T=0.1, eps=1e-9):
    """
    Differentiable surrogate for:  -(#selected) / (#nodes inside the bounding
    hyperrectangle of the selected nodes).

    The hard version is non-differentiable in two places: (1) the box bounds
    (min/max over a thresholded selection) and (2) the inside-count (a step
    function).  Both are smoothed:

      bounds : membership-weighted soft-min/max -- softmax over coordinates
               with log(selection) added, so only selected nodes shape the
               box; -> hard min/max as T -> 0
      count  : soft inside-indicator, product over dimensions of
               sigmoid((p - lo)/T) * sigmoid((hi - p)/T)

    NOTE: gives boundary-only gradients (only the extreme selected nodes move
    the box), so it trains poorly on its own -- prefer soft_scatter_loss.

    selection : (N,) soft selection values in [0, 1]
    pos       : (N, D) descriptor coordinates
    """
    log_m = torch.log(selection + eps)                     # (N,)

    # soft box bounds, per dimension
    w_hi = F.softmax( pos / T + log_m.unsqueeze(1), dim=0) # (N, D)
    w_lo = F.softmax(-pos / T + log_m.unsqueeze(1), dim=0)
    hi   = (w_hi * pos).sum(0)                             # (D,)
    lo   = (w_lo * pos).sum(0)

    # soft inside-count
    inside = (torch.sigmoid((pos - lo) / T)
              * torch.sigmoid((hi - pos) / T)).prod(dim=1) # (N,)

    n_selected = selection.sum()
    n_inside   = inside.sum() + eps

    ratio = n_selected / n_inside
    return -ratio, ratio.item(), n_inside.item()


def soft_hyperellipsoid_loss(selection, pos, level=0.75, ell_T=0.5,
                             eps_reg=1e-4, eps=1e-9):
    """
    Fit a multivariate Gaussian to the (softly) selected nodes via the
    membership-weighted empirical mean and covariance; the q-level ellipsoid
    is {p : maha^2(p) <= chi2_q(D)}, its hard inside-test smoothed by a
    sigmoid of width ell_T:

        s_i = sigmoid((m2_thr - maha_i^2) / ell_T)

    Objective: MAXIMISE the (soft) number of members inside the ellipsoid --
    loss = -sum_i x_i * s_i / sum_i x_i.  Non-members are ignored.

    Note: the member-in fraction is affine-invariant (Mahalanobis distances
    are unchanged by any affine map of the selection), so this rewards
    compact-shaped, thin-tailed selections but puts no pressure on the
    ellipsoid's absolute size or on excluding bystanders.

    selection : (N,) soft selection values in [0, 1]
    pos       : (N, D) descriptor coordinates
    """
    D = pos.size(1)
    m2_thr = float(scipy_stats.chi2.ppf(level, df=D))

    w  = selection / (selection.sum() + eps)                     # (N,)
    mu = (w.unsqueeze(1) * pos).sum(0)                           # (D,)

    diff  = pos - mu                                             # (N, D)
    Sigma = (w.unsqueeze(1) * diff).T @ diff                     # (D, D)
    Sigma = Sigma + eps_reg * torch.eye(D, device=pos.device)

    L    = torch.linalg.cholesky(Sigma)                          # stable inverse
    z    = torch.linalg.solve_triangular(L, diff.T, upper=False) # (D, N)
    maha = (z ** 2).sum(0)                                       # (N,) squared Mahalanobis

    s = torch.sigmoid((m2_thr - maha) / ell_T)                   # soft inside-the-q-ellipsoid

    member_in_mass = (selection * s).sum()                       # members inside
    loss = -member_in_mass / (selection.sum() + eps)
    return loss, member_in_mass.item()

In [ ]:
# Model
in_channels     = feats.shape[1]
hidden_channels = 256
num_mlp_layers  = 3       # depth of the shared per-node encoder MLP

# Training
lr              = 1e-4
num_epochs      = 250
box_T           = 0.1    # temperature of the soft hyperrectangle (active loss)
ell_level       = 0.75   # ellipsoid coverage level (alternative loss / diagnostic)
ell_T           = 0.5    # sigmoid width of the soft ellipsoid boundary
ell_reg         = 1e-4   # covariance ridge for the ellipsoid fit

# Gumbel-Softmax temperature annealing
tau_start       = 1.0
tau_min         = 0.1
anneal_rate     = 0.01   # tau = max(tau_min, tau_start * exp(-anneal_rate * epoch))

In [ ]:
# MLP model consumes descriptors directly -- no graph at all
model = MultiHeadSubsetMLP(
    in_channels=in_channels,
    hidden_channels=hidden_channels,
    num_mlp_layers=num_mlp_layers,
    n_bins=n_bins,
    ell_reg=ell_reg,
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [ ]:
for epoch in range(num_epochs):
    tau = max(tau_min, tau_start * math.exp(-anneal_rate * epoch))
    model.train()
    optimizer.zero_grad()

    # sequential per-bin selection (largest requirement first), each bin conditioned
    # on the Mahalanobis distance to the running mean+covariance of prior selections
    selection, _ = model(feats, pos, bin_node_idx, required, tau=tau)

    # main objective: dense within-cluster scatter (pulls the selection tight)
    loss = soft_scatter_loss(selection, pos)
    # alternatives:
    # loss, ratio, n_inside = soft_hyperrectangle_ratio_loss(selection, pos, T=box_T)
    # loss, mi_mass = soft_hyperellipsoid_loss(selection, pos, level=ell_level,
    #                                          ell_T=ell_T, eps_reg=ell_reg)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"epoch {epoch + 1}/{num_epochs}  tau={tau:.3f}  "
              f"loss={loss.item():.4f}  selected~{int((selection > 0.5).sum())}")

In [ ]:
# Inference: hard straight-through sampling, sequential per-bin selection --
# exactly b_k nodes per bin by construction.
model.eval()
with torch.no_grad():
    selection, logits_all = model(
        feats, pos, bin_node_idx, required, tau=tau_min, hard=True,
    )

selected_nodes = (selection > 0.5).nonzero(as_tuple=True)[0]
print(f"{selected_nodes.numel()} nodes selected "
      f"(total required: {sum(required.values())})")

In [ ]:
# Hard hyperrectangle ratio (exact, non-differentiable -- fine at eval time)
sel_pos    = pos[selected_nodes]
lo_h, hi_h = sel_pos.min(0).values, sel_pos.max(0).values
in_box     = ((pos >= lo_h) & (pos <= hi_h)).all(dim=1)
hard_ratio = selected_nodes.numel() / int(in_box.sum())
print(f"bounding hyperrectangle: lo={lo_h.cpu().numpy().round(3)}  "
      f"hi={hi_h.cpu().numpy().round(3)}")
print(f"hard ratio: {selected_nodes.numel()}/{int(in_box.sum())} = {hard_ratio:.3f}")

In [ ]:
# Hard hyperellipsoid check (exact, non-differentiable -- fine at eval time):
# fit the Gaussian to the hard-selected nodes, count non-members inside the
# ell_level ellipsoid (Mahalanobis^2 <= chi2_{ell_level}(D)) -- the same
# ellipsoid the training loss targets.
D_dim   = pos.size(1)
sel_pos = pos[selected_nodes]

mu_h    = sel_pos.mean(0)
diff_h  = sel_pos - mu_h
Sigma_h = (diff_h.T @ diff_h) / sel_pos.size(0)
Sigma_h = Sigma_h + ell_reg * torch.eye(D_dim, device=pos.device)

L_h    = torch.linalg.cholesky(Sigma_h)
z_all  = torch.linalg.solve_triangular(L_h, (pos - mu_h).T, upper=False)
maha_all = (z_all ** 2).sum(0)                         # (N,)

m2_thr    = scipy_stats.chi2.ppf(ell_level, df=D_dim)
inside    = maha_all <= m2_thr
is_member = torch.zeros_like(inside)
is_member[selected_nodes] = True

nm_inside = int((inside & ~is_member).sum())
m_inside  = int((inside & is_member).sum())
print(f"{ell_level:.0%} ellipsoid (chi2={m2_thr:.2f}, D={D_dim}): "
      f"{m_inside}/{selected_nodes.numel()} members inside, "
      f"{nm_inside} non-members inside")
print(f"member share inside ellipsoid: "
      f"{m_inside / max(m_inside + nm_inside, 1):.3f}")

In [ ]:
# Per-bin selected vs required -- the quota is met by construction
sel_np = selection.cpu().numpy()
rows = []
for bid in sorted(bin_map.keys()):
    idx      = bin_node_idx[bid].cpu().numpy()
    selected = int((sel_np[idx] > 0.5).sum())
    rows.append({
        "bin_id":    bid,
        "available": len(idx),
        "required":  required.get(bid, 0),
        "selected":  selected,
        "met":       selected == required.get(bid, 0),
    })
report = pd.DataFrame(rows)
report.loc["TOTAL"] = ["TOTAL", report["available"].sum(),
                       report["required"].sum(), report["selected"].sum(),
                       bool(report["met"].all())]
display(report)

In [ ]:
# Target-distribution check: subset histogram vs the specified distribution
selected_idx_np = selected_nodes.cpu().numpy()
sel_targets     = df.loc[selected_idx_np, target_col]

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df[target_col], bins=n_bins, alpha=0.35, density=True,
        color="steelblue", label="full dataset")
ax.hist(sel_targets, bins=n_bins, alpha=0.75, density=True,
        color="tomato", label=f"subset (n={len(sel_targets)})")

# overlay the specified pdf
xs = np.linspace(df[target_col].min(), df[target_col].max(), 300)
ax.plot(xs, [pdf(x) for x in xs], "k--", lw=1.5, label="specified pdf")

ax.set_title("Multi-head Gumbel subset — target distribution")
ax.set_xlabel(target_col)
ax.set_ylabel("density")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Interactive view of the selected subset in descriptor space, with the
# bounding HYPER-RECTANGLE of the selected nodes overlaid (matches the active
# loss).  The planted ground-truth group is marked with a distinct colour so
# recovery is readable: gold+red = correctly picked, gold only = missed,
# red only = false positive.
import plotly.graph_objects as go, itertools

sel_mask   = df.index.isin(set(selected_idx_np.tolist()))
planted_np = condition.to_numpy()
cols       = desc_cols
D_plot     = len(cols)

# axis-aligned bounding box of the selected nodes over the plotted columns
sel_coords = df.loc[selected_idx_np, cols].to_numpy()
lo_b = sel_coords.min(0)
hi_b = sel_coords.max(0)

fig = go.Figure()

if D_plot >= 3:
    # planted ground-truth halo (drawn first, behind the point markers)
    fig.add_trace(go.Scatter3d(
        x=df.loc[planted_np, cols[0]], y=df.loc[planted_np, cols[1]], z=df.loc[planted_np, cols[2]],
        mode="markers",
        marker=dict(size=6, color="gold", opacity=0.35),
        name=f"planted (n={int(planted_np.sum())})",
    ))
    fig.add_trace(go.Scatter3d(
        x=df.loc[~sel_mask, cols[0]], y=df.loc[~sel_mask, cols[1]], z=df.loc[~sel_mask, cols[2]],
        mode="markers",
        marker=dict(size=2, color="steelblue", opacity=0.15),
        name="unselected",
    ))
    fig.add_trace(go.Scatter3d(
        x=df.loc[sel_mask, cols[0]], y=df.loc[sel_mask, cols[1]], z=df.loc[sel_mask, cols[2]],
        mode="markers",
        marker=dict(size=3, color="tomato", opacity=0.7),
        name=f"selected (n={sel_mask.sum()})",
    ))

    # box wireframe: the 12 edges of the bounding cuboid (corners differ in one axis)
    corners = list(itertools.product(*zip(lo_b, hi_b)))
    ex, ey, ez = [], [], []
    for a in corners:
        for b in corners:
            if a < b and sum(p != q for p, q in zip(a, b)) == 1:
                ex += [a[0], b[0], None]
                ey += [a[1], b[1], None]
                ez += [a[2], b[2], None]
    fig.add_trace(go.Scatter3d(
        x=ex, y=ey, z=ez, mode="lines",
        line=dict(color="seagreen", width=4),
        name="bounding box",
    ))
    fig.update_layout(
        scene=dict(xaxis_title=cols[0], yaxis_title=cols[1], zaxis_title=cols[2]),
    )
else:
    # planted ground-truth halo (drawn first, behind the point markers)
    fig.add_trace(go.Scatter(
        x=df.loc[planted_np, cols[0]], y=df.loc[planted_np, cols[1]],
        mode="markers",
        marker=dict(size=9, color="gold", opacity=0.35),
        name=f"planted (n={int(planted_np.sum())})",
    ))
    fig.add_trace(go.Scatter(
        x=df.loc[~sel_mask, cols[0]], y=df.loc[~sel_mask, cols[1]],
        mode="markers",
        marker=dict(size=4, color="steelblue", opacity=0.15),
        name="unselected",
    ))
    fig.add_trace(go.Scatter(
        x=df.loc[sel_mask, cols[0]], y=df.loc[sel_mask, cols[1]],
        mode="markers",
        marker=dict(size=5, color="tomato", opacity=0.7),
        name=f"selected (n={sel_mask.sum()})",
    ))

    # bounding rectangle outline
    rx = [lo_b[0], hi_b[0], hi_b[0], lo_b[0], lo_b[0]]
    ry = [lo_b[1], lo_b[1], hi_b[1], hi_b[1], lo_b[1]]
    fig.add_trace(go.Scatter(
        x=rx, y=ry, mode="lines",
        line=dict(color="seagreen", width=2),
        name="bounding box",
    ))
    fig.update_layout(xaxis_title=cols[0], yaxis_title=cols[1])

fig.update_layout(
    title=f"MLP subset — hard box ratio {hard_ratio:.3f}  "
          f"({sel_mask.sum()} selected)",
    width=800, height=650,
    legend=dict(x=0.02, y=0.98),
)

fig.show()